In [1]:
Bài 2 Về Nhà
import heapq

class TSP_AStar:
    def __init__(self, graph):
        self.graph = graph
        self.nodes = list(graph.keys())
        self.num_nodes = len(self.nodes)

    # Heuristic: Trọng số cạnh nhỏ nhất kết nối đỉnh hiện tại với các đỉnh chưa thăm
    def heuristic(self, current, unvisited):
        if not unvisited:
            return self.graph[current].get(self.nodes[0], 0) # Chi phí quay về gốc
        
        min_cost = float('inf')
        for node in unvisited:
            if node in self.graph[current] and self.graph[current][node] < min_cost:
                min_cost = self.graph[current][node]
        return min_cost if min_cost != float('inf') else 0

    def solve(self, start_node):
        # Trạng thái: (f_score, g_score, current_node, visited_nodes_tuple)
        initial_visited = (start_node,)
        open_list = [(0, 0, start_node, initial_visited)]
        
        # Dùng dictionary lưu chi phí để tránh duyệt lại trạng thái y hệt
        best_g_scores = {(start_node, initial_visited): 0}

        while open_list:
            _, current_g, current, visited = heapq.heappop(open_list)

            # Nếu đã thăm đủ đỉnh, quay về gốc
            if len(visited) == self.num_nodes:
                if start_node in self.graph[current]:
                    final_cost = current_g + self.graph[current][start_node]
                    return visited + (start_node,), final_cost
                continue

            unvisited = [n for n in self.nodes if n not in visited]

            for neighbor in unvisited:
                if neighbor in self.graph[current]:
                    move_cost = self.graph[current][neighbor]
                    new_g = current_g + move_cost
                    new_visited = visited + (neighbor,)
                    
                    state_key = (neighbor, new_visited)
                    
                    if new_g < best_g_scores.get(state_key, float('inf')):
                        best_g_scores[state_key] = new_g
                        h = self.heuristic(neighbor, [n for n in self.nodes if n not in new_visited])
                        f = new_g + h
                        heapq.heappush(open_list, (f, new_g, neighbor, new_visited))

        return None, float('inf')

# Kịch bản chạy thử bài toán Giao hàng
if __name__ == "__main__":
    # Đồ thị biểu diễn khoảng cách giữa các điểm giao hàng
    tsp_graph = {
        'A': {'B': 10, 'C': 15, 'D': 20},
        'B': {'A': 10, 'C': 35, 'D': 25},
        'C': {'A': 15, 'B': 35, 'D': 30},
        'D': {'A': 20, 'B': 25, 'C': 30}
    }
    
    tsp_solver = TSP_AStar(tsp_graph)
    best_route, min_cost = tsp_solver.solve('A')
    
    print(f"Lộ trình người giao hàng tối ưu: {' -> '.join(best_route)}")
    print(f"Tổng chi phí quãng đường: {min_cost}")

Lộ trình người giao hàng tối ưu: A -> B -> D -> C -> A
Tổng chi phí quãng đường: 80


In [2]:
#Bài 2 trên Lớp
import heapq

class GraphRoutingAStar:
    def __init__(self, adjacency_list, heuristics):
        self.adj_list = adjacency_list
        self.h = heuristics

    def a_star_search(self, start, target):
        # Hàng đợi ưu tiên lưu (f_score, g_score, current_node)
        open_list = [(self.h[start], 0, start)]
        
        # Dictionary lưu đường đi và chi phí ngắn nhất
        came_from = {}
        g_score = {node: float('inf') for node in self.adj_list}
        g_score[start] = 0
        
        closed_set = set()

        while open_list:
            _, current_g, current_node = heapq.heappop(open_list)

            if current_node in closed_set:
                continue

            if current_node == target:
                return self.reconstruct_path(came_from, current_node)

            closed_set.add(current_node)

            # Duyệt các đỉnh kề
            for neighbor, weight in self.adj_list[current_node]:
                tentative_g_score = current_g + weight

                # Nếu tìm được đường đi tốt hơn
                if tentative_g_score < g_score.get(neighbor, float('inf')):
                    came_from[neighbor] = current_node
                    g_score[neighbor] = tentative_g_score
                    f_score = tentative_g_score + self.h.get(neighbor, 0)
                    heapq.heappush(open_list, (f_score, tentative_g_score, neighbor))

        return "Không tồn tại đường đi!"

    def reconstruct_path(self, came_from, current):
        path = [current]
        while current in came_from:
            current = came_from[current]
            path.append(current)
        return path[::-1]

# Kịch bản chạy thử đồ thị
if __name__ == "__main__":
    graph_data = {
        'A': [('B', 1), ('C', 3), ('D', 7)],
        'B': [('C', 1), ('D', 5)],
        'C': [('D', 12)],
        'D': []
    }
    
    heuristic_data = {'A': 3, 'B': 2, 'C': 1, 'D': 0}
    
    router = GraphRoutingAStar(graph_data, heuristic_data)
    result = router.a_star_search('A', 'D')
    print("Đường đi ngắn nhất từ A đến D là:", " -> ".join(result))


Đường đi ngắn nhất từ A đến D là: A -> B -> D
